<a href="https://colab.research.google.com/github/UnkindledTwo/perceptual_backbone_ldct/blob/main/perceptual_backbone_ldct.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Download and unzip dataset
!gdown 1pIUT5beAkp17WCxyGPHZfHy4kEiJtD4y&export=download
!unzip dataset_biailab_I2I_2025v1.zip

Downloading...
From (original): https://drive.google.com/uc?id=1pIUT5beAkp17WCxyGPHZfHy4kEiJtD4y
From (redirected): https://drive.google.com/uc?id=1pIUT5beAkp17WCxyGPHZfHy4kEiJtD4y&confirm=t&uuid=12617e16-bf62-4aff-87ef-b9670a0f897d
To: /content/dataset_biailab_I2I_2025v1.zip
100% 5.74G/5.74G [01:20<00:00, 71.7MB/s]
Archive:  dataset_biailab_I2I_2025v1.zip
   creating: minideeplesion/
  inflating: __MACOSX/._minideeplesion  
   creating: minideeplesion/test/
  inflating: __MACOSX/minideeplesion/._test  
   creating: minideeplesion/train/
  inflating: __MACOSX/minideeplesion/._train  
   creating: minideeplesion/val/
  inflating: __MACOSX/minideeplesion/._val  
  inflating: minideeplesion/test/001286_01_02_049.png.npz  
  inflating: __MACOSX/minideeplesion/test/._001286_01_02_049.png.npz  
  inflating: minideeplesion/test/001276_01_02_161.png.npz  
  inflating: __MACOSX/minideeplesion/test/._001276_01_02_161.png.npz  
  inflating: minideeplesion/test/001276_04_02_173.png.npz  
  inflati

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
# Clone repo and add to path
!git clone https://github.com/UnkindledTwo/perceptual_backbone_ldct

Cloning into 'perceptual_backbone_ldct'...
remote: Enumerating objects: 28, done.
remote: Counting objects: 100% (28/28), done.
remote: Compressing objects: 100% (25/25), done.
remote: Total 28 (delta 4), reused 20 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (28/28), 53.24 KiB | 5.32 MiB/s, done.
Resolving deltas: 100% (4/4), done.


In [4]:
import torch
from pathlib import Path

import sys
sys.path.append("/content/perceptual_backbone_ldct/src")
sys.path.append("/content/perceptual_backbone_ldct")

from config import PerceptualConfig
CONFIG = PerceptualConfig(run_name="screening-c0", data_root=Path("/content/minideeplesion/"))

In [5]:
from pathlib import Path
from data import assert_no_patient_leakage
patient_leakage = assert_no_patient_leakage(Path(CONFIG.data_root))
print("Patients per split:",patient_leakage)

Patients per split: {'train': 115, 'val': 34, 'test': 31}


In [6]:
import numpy as np
import matplotlib.pyplot as plt
from glob import glob

train_file_path = CONFIG.data_root.as_posix() + '/train/*.npz'
test_file_path = CONFIG.data_root.as_posix() + '/test/*.npz'
val_file_path = CONFIG.data_root.as_posix() + '/val/*.npz'

# Find all files
train_files = sorted(glob(train_file_path))
test_files = sorted(glob(test_file_path))
val_files = sorted(glob(val_file_path))

print(f"Training samples: {len(train_files)}")
print(f"Test samples: {len(test_files)}")
print(f"Eval samples: {len(val_files)}")

# Load a single file
data = np.load(train_files[0])
print(f"Available arrays: {data.files}")

# Access the arrays
gt_image = data[CONFIG.target_key]
noisy_image = data[CONFIG.input_key]

print(f"Image shape: {gt_image.shape}")
print(f"Value range: [{gt_image.min():.3f}, {gt_image.max():.3f}]")

Training samples: 899
Test samples: 220
Eval samples: 182
Available arrays: ['gt_full', 'gt_clipped', 'noisy_clipped_10k', 'noisy_clipped_50k', 'noisy_clipped_100k', 'noisy_clipped_500k', 'noisy_clipped_1000k']
Image shape: (512, 512)
Value range: [0.000, 1.000]


In [7]:
from data import I2IPairs
from torch.utils.data import DataLoader

# Dataloaders
splits = {s: I2IPairs(Path(CONFIG.data_root / s), CONFIG.input_key, CONFIG.target_key) for s in ("train", "val", "test")}
train_loader = DataLoader(splits["train"], batch_size=CONFIG.batch_size, shuffle=True, num_workers=4, pin_memory=True)
test_loader = DataLoader(splits["test"], batch_size=CONFIG.batch_size, shuffle=True, num_workers=4, pin_memory=True)
val_loader = DataLoader(splits["val"], batch_size=CONFIG.batch_size, shuffle=True, num_workers=4, pin_memory=True)

print(f"Train batches: {len(train_loader)}")
print(f"Test batches: {len(test_loader)}")
print(f"Val batches: {len(val_loader)}")

Train batches: 113
Test batches: 28
Val batches: 23


In [8]:
!pip install lpips piq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.8/53.8 kB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 106.9/106.9 kB 2.3 MB/s eta 0:00:00


In [9]:
# Identity test for psnr, ssim, lpips-alex, dists and R_h
from metrics import slice_metrics, save_per_slice_metrics

noisy, gt, _ = next(iter((test_loader)))

noisy_2d = noisy[0].squeeze().cpu().numpy()
gt_2d = gt[0].squeeze().cpu().numpy()

# Run the metrics on this single slice
metrics_noisy = slice_metrics(noisy_2d, gt_2d)
metrics_gt = slice_metrics(gt_2d, gt_2d)

print(metrics_noisy)
print(metrics_gt)

Setting up [LPIPS] perceptual loss: trunk [alex], v[0.1], spatial [off]
Downloading: "https://download.pytorch.org/models/alexnet-owt-7be5be79.pth" to /root/.cache/torch/hub/checkpoints/alexnet-owt-7be5be79.pth


/usr/local/lib/python3.13/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=AlexNet_Weights.IMAGENET1K_V1`. You can also use `weights=AlexNet_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
100%|██████████| 233M/233M [00:00<00:00, 254MB/s]


Loading model from: /usr/local/lib/python3.13/dist-packages/lpips/weights/v0.1/alex.pth
Downloading: "https://github.com/photosynthesis-team/piq/releases/download/v0.4.1/dists_weights.pt" to /root/.cache/torch/hub/checkpoints/dists_weights.pt


/usr/local/lib/python3.13/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG16_Weights.IMAGENET1K_V1`. You can also use `weights=VGG16_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/vgg16-397923af.pth" to /root/.cache/torch/hub/checkpoints/vgg16-397923af.pth
{'psnr': 21.47908218197129, 'ssim': 0.7311966549230524, 'lpips': 0.2768171429634094, 'dists': 0.1749798059463501, 'r_h': 6.471908291282494}
{'psnr': inf, 'ssim': 1.0, 'lpips': 0.0, 'dists': 1.1920928955078125e-07, 'r_h': 1.0}


/usr/local/lib/python3.13/dist-packages/skimage/metrics/simple_metrics.py:168: RuntimeWarning: divide by zero encountered in scalar divide
  return 10 * np.log10((data_range**2) / err)


In [10]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [11]:
from model import UNet

# Init U-Net

model = UNet().to(device)
model.eval()

UNet(
  (encoder): ModuleList(
    (0): DoubleConv(
      (conv): Sequential(
        (0): Conv2d(1, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (2): ReLU(inplace=True)
        (3): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (4): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (5): ReLU(inplace=True)
      )
    )
    (1): DoubleConv(
      (conv): Sequential(
        (0): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (1): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (2): ReLU(inplace=True)
        (3): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (4): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (5): ReLU(inplace

In [12]:
from train import train_and_test
import time
import random

print("Seed:", CONFIG.seed)
random.seed(CONFIG.seed)

criterion = torch.nn.functional.l1_loss

start_time = time.time()
df_test = train_and_test(config=CONFIG, model=model, criterion=criterion, device=device)
end_time = time.time()
total_time = end_time - start_time
print("Total time", total_time)
print(f"Total training time: {total_time:.2f} seconds")

Setting up [LPIPS] perceptual loss: trunk [alex], v[0.1], spatial [off]
Loading model from: /usr/local/lib/python3.13/dist-packages/lpips/weights/v0.1/alex.pth
Seed: 42


screening-c0:   0%|          | 0/30 [00:02<?, ?it/s]


KeyboardInterrupt: 

In [ ]:
summary = {
    "psnr_pred_mean":  float(df_test["psnr"].mean()),
    "psnr_pred_std":   float(df_test["psnr"].std(ddof=1)),
    "ssim_pred_mean":  float(df_test["ssim"].mean()),
    "ssim_pred_std":   float(df_test["ssim"].std(ddof=1)),
    "lpips_pred_mean":  float(df_test["lpips"].mean()),
    "lpips_pred_std":   float(df_test["lpips"].std(ddof=1)),
    "dists_pred_mean":  float(df_test["dists"].mean()),
    "dists_pred_std":   float(df_test["dists"].std(ddof=1)),
    "r_h_pred_mean":  float(df_test["r_h"].mean()),
    "r_h_pred_std":   float(df_test["r_h"].std(ddof=1)),
    "n_test": int(len(df_test)),
    "total_time": total_time,
}

print("\n=== TEST RESULTS (mean \u00b1 std) ===")
print(f"Model  PSNR: {summary['psnr_pred_mean']:.2f} \u00b1 {summary['psnr_pred_std']:.2f} dB")
print(f"Model  SSIM: {summary['ssim_pred_mean']:.4f} \u00b1 {summary['ssim_pred_std']:.4f}")
print(f"Model  LP\u0130PS: {summary['lpips_pred_mean']:.4f} \u00b1 {summary['lpips_pred_std']:.4f}")
print(f"Model  DISTS: {summary['dists_pred_mean']:.4f} \u00b1 {summary['dists_pred_std']:.4f}")
print(f"Model  R_H: {summary['r_h_pred_mean']:.4f} \u00b1 {summary['r_h_pred_std']:.4f}")

print(f"Test samples: {summary['n_test']}")

save_per_slice_metrics([summary], CONFIG.run_dir / "summary.csv")

### Evaluating Baseline Metrics (Noisy vs Ground Truth)

In [16]:
import pandas as pd
from tqdm import tqdm
from metrics import slice_metrics

baseline_metrics = []

# Iterate through the test loader to calculate metrics on raw inputs
for noisy_batch, gt_batch, _ in tqdm(test_loader, desc="Evaluating Baseline"):
    for i in range(noisy_batch.size(0)):
        noisy_2d = noisy_batch[i].squeeze().cpu().numpy()
        gt_2d = gt_batch[i].squeeze().cpu().numpy()

        # Compute metrics between noisy input and ground truth target
        metrics = slice_metrics(noisy_2d, gt_2d)
        baseline_metrics.append(metrics)

# Convert to DataFrame and summarize
df_baseline = pd.DataFrame(baseline_metrics)
save_per_slice_metrics(df_baseline, CONFIG.run_dir / "baseline_metrics.csv")
baseline_summary = {
    "psnr_raw_mean":  float(df_baseline["psnr"].mean()),
    "psnr_raw_std":   float(df_baseline["psnr"].std(ddof=1)),
    "ssim_raw_mean":  float(df_baseline["ssim"].mean()),
    "ssim_raw_std":   float(df_baseline["ssim"].std(ddof=1)),
    "lpips_raw_mean":  float(df_baseline["lpips"].mean()),
    "lpips_raw_std":   float(df_baseline["lpips"].std(ddof=1)),
    "dists_raw_mean":  float(df_baseline["dists"].mean()),
    "dists_raw_std":   float(df_baseline["dists"].std(ddof=1)),
    "r_h_raw_mean":  float(df_baseline["r_h"].mean()),
    "r_h_raw_std":   float(df_baseline["r_h"].std(ddof=1)),
    "n_test": len(df_baseline)
}

print("\n=== RAW NOISY INPUT TEST RESULTS (mean ± std) ===")
print(f"Raw PSNR: {baseline_summary['psnr_raw_mean']:.2f} ± {baseline_summary['psnr_raw_std']:.2f} dB")
print(f"Raw SSIM: {baseline_summary['ssim_raw_mean']:.4f} ± {baseline_summary['ssim_raw_std']:.4f}")
print(f"Raw LPIPS: {baseline_summary['lpips_raw_mean']:.4f} ± {baseline_summary['lpips_raw_std']:.4f}")
print(f"Raw DISTS: {baseline_summary['dists_raw_mean']:.4f} ± {baseline_summary['dists_raw_std']:.4f}")
print(f"Raw R_H: {baseline_summary['r_h_raw_mean']:.4f} ± {baseline_summary['r_h_raw_std']:.4f}")

save_per_slice_metrics([baseline_summary], CONFIG.run_dir / "baseline_summary.csv")



=== RAW NOISY INPUT TEST RESULTS (mean ± std) ===
Raw PSNR: 23.04 ± 2.13 dB
Raw SSIM: 0.7423 ± 0.0647
Raw LPIPS: 0.2843 ± 0.0959
Raw DISTS: 0.1641 ± 0.0462
Raw R_H: 6.1743 ± 5.8560


,psnr_raw_mean,psnr_raw_std,ssim_raw_mean,ssim_raw_std,lpips_raw_mean,lpips_raw_std,dists_raw_mean,dists_raw_std,r_h_raw_mean,r_h_raw_std,n_test
0,23.039831,2.134886,0.742302,0.064748,0.284252,0.095883,0.164106,0.046221,6.174314,5.856038,220


In [ ]:
from train import only_test
"""
df_test = only_test(CONFIG, model, CONFIG.run_dir/"scoped.pth", device)

summary = {
    "psnr_pred_mean":  float(df_test["psnr"].mean()),
    "psnr_pred_std":   float(df_test["psnr"].std(ddof=1)),
    "ssim_pred_mean":  float(df_test["ssim"].mean()),
    "ssim_pred_std":   float(df_test["ssim"].std(ddof=1)),
    "lpips_pred_mean":  float(df_test["lpips"].mean()),
    "lpips_pred_std":   float(df_test["lpips"].std(ddof=1)),
    "dists_pred_mean":  float(df_test["dists"].mean()),
    "dists_pred_std":   float(df_test["dists"].std(ddof=1)),
    "r_h_pred_mean":  float(df_test["r_h"].mean()),
    "r_h_pred_std":   float(df_test["r_h"].std(ddof=1)),
    "n_test": int(len(df_test)),
    "total_time": total_time,
}

print("\n=== TEST RESULTS (mean \u00b1 std) ===")
print(f"Model  PSNR: {summary['psnr_pred_mean']:.2f} \u00b1 {summary['psnr_pred_std']:.2f} dB")
print(f"Model  SSIM: {summary['ssim_pred_mean']:.4f} \u00b1 {summary['ssim_pred_std']:.4f}")
print(f"Model  LP\u0130PS: {summary['lpips_pred_mean']:.4f} \u00b1 {summary['lpips_pred_std']:.4f}")
print(f"Model  DISTS: {summary['dists_pred_mean']:.4f} \u00b1 {summary['dists_pred_std']:.4f}")
print(f"Model  R_H: {summary['r_h_pred_mean']:.4f} \u00b1 {summary['r_h_pred_std']:.4f}")

print(f"Test samples: {summary['n_test']}")

save_per_slice_metrics([summary], CONFIG.run_dir / "summary_scoped.csv")
"""

In [13]:
!pip install -q huggingface_hub

In [14]:
from huggingface_hub import hf_hub_download
import huggingface_hub
import os

huggingface_hub.login()

radimagenet_weights_path = hf_hub_download(repo_id="Lab-Rasool/RadImageNet", filename="ResNet50.pt")
print(f"Successfully downloaded RadImageNet weights to: {radimagenet_weights_path}")

ResNet50.pt: reconstructing file:   0%|          |  0.00B / 94.3MB            

ResNet50.pt: downloading bytes:           |  0.00B            

Successfully downloaded RadImageNet weights to: /root/.cache/huggingface/hub/models--Lab-Rasool--RadImageNet/snapshots/14460ee4c1276f6925611a63aa9a54a05d39eae0/ResNet50.pt


In [40]:
from feature_nets import build_feature_net

random_tensor = torch.rand(64, 64)
random_tensor = random_tensor.unsqueeze(0).unsqueeze(0)

vggnet = build_feature_net("vgg19")
radimagenet_net = build_feature_net("radimagenet", radimagenet_weights_path)
dinov2_s_net = build_feature_net("dinov2_s")
dinov2_b_net = build_feature_net("dinov2_b")
dinov2_l_net = build_feature_net("dinov2_l")
medsiglipnet = build_feature_net("medsiglip")
clip_b16net = build_feature_net("clip_b16")

from distances import PiqDistance
lpips_distance = PiqDistance("lpips_vgg")
dists_distance = PiqDistance("dists")

networks = {
    "VGG19": vggnet,
    "RadImageNet": radimagenet_net,
    "DINOv2_S": dinov2_s_net,
    "DINOv2_B": dinov2_b_net,
    "DINOv2_L": dinov2_l_net,
    "MedSigLIP": medsiglipnet,
    "CLIP_B16": clip_b16net
}

Using cache found in /root/.cache/torch/hub/facebookresearch_dinov2_main
Using cache found in /root/.cache/torch/hub/facebookresearch_dinov2_main
Using cache found in /root/.cache/torch/hub/facebookresearch_dinov2_main


Loading weights:   0%|          | 0/888 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] CLIPVisionModel LOAD REPORT from: openai/clip-vit-base-patch16
Key                                                          | Status     |  | 
-------------------------------------------------------------+------------+--+-
text_model.encoder.layers.{0...11}.self_attn.v_proj.bias     | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm1.weight        | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm2.weight        | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc2.bias              | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.k_proj.weight   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.q_proj.bias     | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.out_proj.weight | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.k_proj.bias     | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.out_proj.bias   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc2

In [19]:
import pandas as pd
import time
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

results = []
batch_size = 8
img_size = 224

print(f"Benchmarking on {device}...")

for name, net in networks.items():
    net = net.to(device)
    net.eval()

    # 1. Distance when pred == target
    pred_ident = torch.rand(1, 1, img_size, img_size, device=device)
    target_ident = pred_ident.clone()
    feat_pred = net.features(pred_ident)
    feat_target = net.features(target_ident)

    dist_ident_pw = distances.pointwise_distance(feat_pred, feat_target).item()
    dist_ident_stat = distances.statistics_distance(feat_pred, feat_target).item()

    # 2a. Gradient check (Pointwise)
    pred_pw = torch.rand(1, 1, img_size, img_size, requires_grad=True, device=device)
    target_pw = torch.rand(1, 1, img_size, img_size, device=device)
    loss_pw = distances.pointwise_distance(net.features(pred_pw), net.features(target_pw))
    loss_pw.backward()
    has_grad_pw = pred_pw.grad is not None and pred_pw.grad.abs().sum().item() > 0

    # 2b. Gradient check (Statistics)
    pred_stat = torch.rand(1, 1, img_size, img_size, requires_grad=True, device=device)
    target_stat = torch.rand(1, 1, img_size, img_size, device=device)
    loss_stat = distances.statistics_distance(net.features(pred_stat), net.features(target_stat))
    loss_stat.backward()
    has_grad_stat = pred_stat.grad is not None and pred_stat.grad.abs().sum().item() > 0

    # 3. Speed test (ms per batch of 8)
    try:
        # Warmup
        for _ in range(2):
            p_w = torch.rand(batch_size, 1, img_size, img_size, requires_grad=True, device=device)
            t_w = torch.rand(batch_size, 1, img_size, img_size, device=device)
            f_p = net.features(p_w)
            f_t = net.features(t_w)
            distances.pointwise_distance(f_p, f_t).backward(retain_graph=True)
            distances.statistics_distance(f_p, f_t).backward()

        n_iters = 5
        start_events_pw = [torch.cuda.Event(enable_timing=True) for _ in range(n_iters)]
        end_events_pw = [torch.cuda.Event(enable_timing=True) for _ in range(n_iters)]

        start_events_stat = [torch.cuda.Event(enable_timing=True) for _ in range(n_iters)]
        end_events_stat = [torch.cuda.Event(enable_timing=True) for _ in range(n_iters)]

        for i in range(n_iters):
            # Timing Pointwise
            p_bench_pw = torch.rand(batch_size, 1, img_size, img_size, requires_grad=True, device=device)
            t_bench_pw = torch.rand(batch_size, 1, img_size, img_size, device=device)
            start_events_pw[i].record()
            loss_p = distances.pointwise_distance(net.features(p_bench_pw), net.features(t_bench_pw))
            loss_p.backward()
            end_events_pw[i].record()

            # Timing Statistics
            p_bench_stat = torch.rand(batch_size, 1, img_size, img_size, requires_grad=True, device=device)
            t_bench_stat = torch.rand(batch_size, 1, img_size, img_size, device=device)
            start_events_stat[i].record()
            loss_s = distances.statistics_distance(net.features(p_bench_stat), net.features(t_bench_stat))
            loss_s.backward()
            end_events_stat[i].record()

        torch.cuda.synchronize()
        times_pw = [s.elapsed_time(e) for s, e in zip(start_events_pw, end_events_pw)]
        avg_time_pw = sum(times_pw) / n_iters

        times_stat = [s.elapsed_time(e) for s, e in zip(start_events_stat, end_events_stat)]
        avg_time_stat = sum(times_stat) / n_iters

    except RuntimeError as e:
        if "out of memory" in str(e).lower():
            avg_time_pw = "OOM"
            avg_time_stat = "OOM"
            torch.cuda.empty_cache()
        else:
            avg_time_pw = f"Error"
            avg_time_stat = f"Error"
            print(f"{name} Error: {e}")

    results.append({
        "Network": name,
        "PW Dist (pred=tgt)": f"{dist_ident_pw:.6f}",
        "PW Dist ~0": dist_ident_pw < 1e-5,
        "PW Grad": has_grad_pw,
        "PW Time (ms)": avg_time_pw,
        "Stat Dist (pred=tgt)": f"{dist_ident_stat:.6f}",
        "Stat Dist ~0": dist_ident_stat < 1e-5,
        "Stat Grad": has_grad_stat,
        "Stat Time (ms)": avg_time_stat
    })

    # Move net back to CPU and clear cache to avoid OOM for subsequent models
    net = net.cpu()
    torch.cuda.empty_cache()


df_benchmark = pd.DataFrame(results)
display(df_benchmark)
save_per_slice_metrics(df_benchmark, CONFIG.run_dir / "benchmark_distances.csv")

Benchmarking on cuda...


,Network,PW Dist (pred=tgt),PW Dist ~0,PW Grad,PW Time (ms),Stat Dist (pred=tgt),Stat Dist ~0,Stat Grad,Stat Time (ms)
0,VGG19,0.000000,True,True,13.865933,0.000000,True,False,14.447123
1,RadImageNet,0.000000,True,True,13.801299,0.000000,True,False,14.940442
2,DINOv2_S,0.000000,True,True,26.722272,0.000000,True,True,27.435476
3,DINOv2_B,0.000000,True,True,58.330566,0.000000,True,True,59.091508
4,DINOv2_L,0.000000,True,True,152.826074,0.000000,True,True,153.670508
5,MedSigLIP,0.000000,True,True,853.450720,0.000000,True,True,855.970544
6,CLIP_B16,0.000000,True,True,42.733971,0.000000,True,True,43.486611


In [42]:
import pandas as pd
import time
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Benchmarking PiqDistance metrics on {device}...")

piq_metrics = {
    "LPIPS (VGG)": lpips_distance,
    "DISTS": dists_distance
}

piq_results = []
batch_size = 8
img_size = 224

for name, metric in piq_metrics.items():
    # 1. Distance when pred == target
    # PiqDistance typically expects [B, C, H, W]
    pred_ident = torch.rand(1, 1, img_size, img_size, device=device)
    target_ident = pred_ident.clone()

    dist_ident = metric(pred_ident, target_ident).item()

    # 2. Gradient check
    pred_grad = torch.rand(1, 1, img_size, img_size, requires_grad=True, device=device)
    target_grad = torch.rand(1, 1, img_size, img_size, device=device)
    loss = metric(pred_grad, target_grad)
    loss.backward()
    has_grad = pred_grad.grad is not None and pred_grad.grad.abs().sum().item() > 0

    # 3. Speed test (ms per batch of 8)
    try:
        # Warmup
        for _ in range(2):
            p_w = torch.rand(batch_size, 1, img_size, img_size, requires_grad=True, device=device)
            t_w = torch.rand(batch_size, 1, img_size, img_size, device=device)
            metric(p_w, t_w).backward()

        n_iters = 5
        start_events = [torch.cuda.Event(enable_timing=True) for _ in range(n_iters)]
        end_events = [torch.cuda.Event(enable_timing=True) for _ in range(n_iters)]

        for i in range(n_iters):
            p_bench = torch.rand(batch_size, 1, img_size, img_size, requires_grad=True, device=device)
            t_bench = torch.rand(batch_size, 1, img_size, img_size, device=device)

            start_events[i].record()
            loss_bench = metric(p_bench, t_bench)
            loss_bench.backward()
            end_events[i].record()

        torch.cuda.synchronize()
        times = [s.elapsed_time(e) for s, e in zip(start_events, end_events)]
        avg_time = sum(times) / n_iters

    except RuntimeError as e:
        if "out of memory" in str(e).lower():
            avg_time = "OOM"
            torch.cuda.empty_cache()
        else:
            avg_time = f"Error: {e}"

    piq_results.append({
        "Metric": name,
        "Identity Dist (pred=tgt)": f"{dist_ident:.6f}",
        "Identity ~0": dist_ident < 1e-5,
        "Gradients Functional": has_grad,
        "Time (ms per batch of 8)": avg_time
    })

df_piq_benchmark = pd.DataFrame(piq_results)
display(df_piq_benchmark)
save_per_slice_metrics(df_benchmark, CONFIG.run_dir / "benchmark_distances_lpips_dists.csv")

Benchmarking PiqDistance metrics on cuda...


,Metric,Identity Dist (pred=tgt),Identity ~0,Gradients Functional,Time (ms per batch of 8)
0,LPIPS (VGG),0.000000,True,True,15.988326
1,DISTS,0.000000,True,True,17.515316


,Network,PW Dist (pred=tgt),PW Dist ~0,PW Grad,PW Time (ms),Stat Dist (pred=tgt),Stat Dist ~0,Stat Grad,Stat Time (ms)
0,VGG19,0.000000,True,True,13.865933,0.000000,True,False,14.447123
1,RadImageNet,0.000000,True,True,13.801299,0.000000,True,False,14.940442
2,DINOv2_S,0.000000,True,True,26.722272,0.000000,True,True,27.435476
3,DINOv2_B,0.000000,True,True,58.330566,0.000000,True,True,59.091508
4,DINOv2_L,0.000000,True,True,152.826074,0.000000,True,True,153.670508
5,MedSigLIP,0.000000,True,True,853.450720,0.000000,True,True,855.970544
6,CLIP_B16,0.000000,True,True,42.733971,0.000000,True,True,43.486611


In [49]:
from screen import screen_network
from pathlib import Path
import torch.nn.functional as F



                 slice_id  ordering   d_10000   d_50000  d_100000  d_500000  \
0    000002_01_01_162.png       1.0  0.287876  0.161818  0.119108  0.060049   
1    000002_01_01_176.png       1.0  0.282949  0.159412  0.119778  0.061120   
2    000002_02_01_050.png       1.0  0.265227  0.141935  0.102681  0.049152   
3    000002_02_01_052.png       1.0  0.264336  0.143067  0.104182  0.049191   
4    000002_02_01_065.png       1.0  0.266362  0.146132  0.106187  0.051416   
..                    ...       ...       ...       ...       ...       ...   
177  001305_01_01_056.png       1.0  0.077116  0.039748  0.030926  0.021116   
178  001312_01_01_066.png       1.0  0.095428  0.048964  0.035204  0.018348   
179  001312_02_01_014.png       1.0  0.070379  0.035718  0.025368  0.012329   
180  001312_02_02_209.png       1.0  0.085303  0.058210  0.053894  0.049886   
181  001312_03_01_077.png       1.0  0.089707  0.046128  0.033858  0.017167   

     d_1000000    d_blur  
0     0.046999  0.081620

KeyboardInterrupt: 

### Comprehensive Metric Screening for Feature Networks
We will now run the screening across the validation set for the requested metrics and network backbones, computing:
1. **Ordering Score ($M_\phi$)**: Spearman's rank correlation over the 5 dose levels.
2. **Dose Ratio ($D_\phi(10k) / D_\phi(1000k)$)**: Indicating distance scale change across dose variations.
3. **Blur Penalty ($B_\phi$)**: Evaluating sensitivity to blur relative to equivalent-MSE noise levels.

In [50]:
import numpy as np
import pandas as pd
import torch
from pathlib import Path
from screen import screen_network
import distances

val_paths = [Path(f) for f in val_files]

# Helper to extract and count network parameters
def get_param_count(model):
    if model is None:
        return 0
    return sum(p.numel() for p in model.parameters())

# We will collect final statistics for each network/matching configuration
screen_table_results = []

# 1. Pixel L1 Reference
def pixel_l1_distance(x, y):
    return torch.mean(torch.abs(x - y))

print("Screening Pixel L1 (Reference)...")
df_l1 = screen_network(pixel_l1_distance, val_paths, device)
l1_m_phi = df_l1["ordering"].mean()
l1_ratio = df_l1["d_10000"].mean() / (df_l1["d_1000000"].mean() + 1e-8)
l1_b_phi = df_l1["d_blur"].sum() / (df_l1["d_100000"].sum() + 1e-8)

screen_table_results.append({
    "Network": "Pixel L1 (reference)",
    "Matching": "-",
    "Ordering M_phi": f"{l1_m_phi:.4f}",
    "D_10k / D_1000k": f"{l1_ratio:.4f}",
    "Blur penalty B_phi": f"{l1_b_phi:.4f}",
    "Parameters": "0"
})

# 2. Setup evaluations for all model and distance pairings
configs = [
    {"name": "VGG-19 relu3_2", "net": vggnet, "matching": "pointwise", "dist_fn": distances.pointwise_distance},
    {"name": "LPIPS-VGG", "net": None, "matching": "pointwise", "dist_fn": lpips_distance},
    {"name": "DISTS", "net": None, "matching": "statistics", "dist_fn": dists_distance},
    {"name": "RadImageNet ResNet-50", "net": radimagenet_net, "matching": "pointwise", "dist_fn": distances.pointwise_distance},
    {"name": "DINOv2-S", "net": dinov2_s_net, "matching": "pointwise", "dist_fn": distances.pointwise_distance},
    {"name": "DINOv2-S", "net": dinov2_s_net, "matching": "statistics", "dist_fn": distances.statistics_distance},
    {"name": "DINOv2-B", "net": dinov2_b_net, "matching": "pointwise", "dist_fn": distances.pointwise_distance},
    {"name": "DINOv2-L", "net": dinov2_l_net, "matching": "pointwise", "dist_fn": distances.pointwise_distance},
    {"name": "CLIP ViT-B/16", "net": clip_b16net, "matching": "pointwise", "dist_fn": distances.pointwise_distance},
    {"name": "MedSigLIP-448", "net": medsiglipnet, "matching": "pointwise", "dist_fn": distances.pointwise_distance}
]

for cfg in configs:
    print(f"Screening {cfg['name']} ({cfg['matching']})...")
    net = cfg["net"]
    dist_fn = cfg["dist_fn"]

    # Calculate parameter count
    param_count = "-" if net is None else f"{get_param_count(net):,}"

    if net is not None:
        net = net.to(device)
        net.eval()
        # Wrap distance with feature extraction
        def wrapped_distance(x, y):
            with torch.no_grad():
                return dist_fn(net.features(x), net.features(y))
        eval_fn = wrapped_distance
    else:
        # Directly callable metric (LPIPS/DISTS)
        eval_fn = dist_fn

    try:
        df_res = screen_network(eval_fn, val_paths, device)
        m_phi = df_res["ordering"].mean()
        ratio = df_res["d_10000"].mean() / (df_res["d_1000000"].mean() + 1e-8)
        b_phi = df_res["d_blur"].sum() / (df_res["d_100000"].sum() + 1e-8)

        screen_table_results.append({
            "Network": cfg["name"],
            "Matching": cfg["matching"],
            "Ordering M_phi": f"{m_phi:.4f}",
            "D_10k / D_1000k": f"{ratio:.4f}",
            "Blur penalty B_phi": f"{b_phi:.4f}",
            "Parameters": param_count
        })
    except Exception as e:
        print(f"Failed to evaluate {cfg['name']}: {e}")
        screen_table_results.append({
            "Network": cfg["name"],
            "Matching": cfg["matching"],
            "Ordering M_phi": "Error",
            "D_10k / D_1000k": "Error",
            "Blur penalty B_phi": "Error",
            "Parameters": param_count
        })

    if net is not None:
        net = net.cpu()
        torch.cuda.empty_cache()

# Create summary table
df_screen_report = pd.DataFrame(screen_table_results)
display(df_screen_report)
save_per_slice_metrics(df_screen_report, CONFIG.run_dir / "zoo_screen_report.csv")

Screening Pixel L1 (Reference)...


Screening VGG-19 relu3_2 (pointwise)...


Screening LPIPS-VGG (pointwise)...


Screening DISTS (statistics)...


Screening RadImageNet ResNet-50 (pointwise)...


...

Screening DINOv2-L (pointwise)...


Screening CLIP ViT-B/16 (pointwise)...


Screening MedSigLIP-448 (pointwise)...


,Network,Matching,Ordering M_phi,D_10k / D_1000k,Blur penalty B_phi,Parameters
0,Pixel L1 (reference),-,1.0000,5.7366,0.8861,0
1,VGG-19 relu3_2,pointwise,1.0000,2.6057,1.5091,"1,145,408"
2,LPIPS-VGG,pointwise,0.9984,2.4252,1.4826,-
3,DISTS,statistics,0.9835,2.7510,1.4495,-
4,RadImageNet ResNet-50,pointwise,0.9995,7.2219,1.1735,"25,557,032"
5,DINOv2-S,pointwise,0.9978,7.7106,1.7517,"22,056,576"
6,DINOv2-S,statistics,0.9973,2.3665,1.2603,"22,056,576"
7,DINOv2-B,pointwise,0.9973,6.2701,1.6693,"86,580,480"
8,DINOv2-L,pointwise,0.9956,6.8855,1.8737,"304,368,640"
9,CLIP ViT-B/16,pointwise,0.9956,2.7333,1.5709,"85,799,424"


,Network,Matching,Ordering M_phi,D_10k / D_1000k,Blur penalty B_phi,Parameters
0,Pixel L1 (reference),-,1.0000,5.7366,0.8861,0
1,VGG-19 relu3_2,pointwise,1.0000,2.6057,1.5091,"1,145,408"
2,LPIPS-VGG,pointwise,0.9984,2.4252,1.4826,-
3,DISTS,statistics,0.9835,2.7510,1.4495,-
4,RadImageNet ResNet-50,pointwise,0.9995,7.2219,1.1735,"25,557,032"
5,DINOv2-S,pointwise,0.9978,7.7106,1.7517,"22,056,576"
6,DINOv2-S,statistics,0.9973,2.3665,1.2603,"22,056,576"
7,DINOv2-B,pointwise,0.9973,6.2701,1.6693,"86,580,480"
8,DINOv2-L,pointwise,0.9956,6.8855,1.8737,"304,368,640"
9,CLIP ViT-B/16,pointwise,0.9956,2.7333,1.5709,"85,799,424"
